# 01 Data Audit: SaaS Sales
Goal: check the raw data quality, create new columns, save a clean file.
Source: Amazon AWS SaaS Sales Dataset (Kaggle), a fictional B2B SaaS company. Licence: GFDL.

In [1]:
import os
if os.path.basename(os.getcwd()) == "Notebooks":
    os.chdir("..")
print(os.getcwd())

C:\Saas-Sales-Profitability


In [2]:
import pandas as pd
df = pd.read_csv("Data/Raw/SaaS-Sales.csv")
print(df.shape)
print(df.columns.tolist())
df.head()


(9994, 19)
['Row ID', 'Order ID', 'Order Date', 'Date Key', 'Contact Name', 'Country', 'City', 'Region', 'Subregion', 'Customer', 'Customer ID', 'Industry', 'Segment', 'Product', 'License', 'Sales', 'Quantity', 'Discount', 'Profit']


,Row ID,Order ID,Order Date,Date Key,Contact Name,Country,City,Region,Subregion,Customer,Customer ID,Industry,Segment,Product,License,Sales,Quantity,Discount,Profit
0,1,EMEA-2022-152156,11/9/2022,20221109,Nathan Bell,Ireland,Dublin,EMEA,UKIR,Chevron,1017,Energy,SMB,Marketing Suite,16GRM07R1K,261.9600,2,0.00,41.9136
1,2,EMEA-2022-152156,11/9/2022,20221109,Nathan Bell,Ireland,Dublin,EMEA,UKIR,Chevron,1017,Energy,SMB,FinanceHub,QLIW57KZUV,731.9400,3,0.00,219.5820
2,3,AMER-2022-138688,6/13/2022,20220613,Deirdre Bailey,United States,New York City,AMER,NAMER,Phillips 66,1056,Energy,Strategic,FinanceHub,JI6BVL70HQ,14.6200,2,0.00,6.8714
3,4,EMEA-2021-108966,10/11/2021,20211011,Zoe Hodges,Germany,Stuttgart,EMEA,EU-WEST,Royal Dutch Shell,1031,Energy,SMB,ContactMatcher,DE9GJKGD44,957.5775,5,0.45,-383.0310
4,5,EMEA-2021-108966,10/11/2021,20211011,Zoe Hodges,Germany,Stuttgart,EMEA,EU-WEST,Royal Dutch Shell,1031,Energy,SMB,Marketing Suite - Gold,OIF7NY23WD,22.3680,2,0.20,2.5164


In [3]:
df.info()
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Row ID        9994 non-null   int64  
 1   Order ID      9994 non-null   object 
 2   Order Date    9994 non-null   object 
 3   Date Key      9994 non-null   int64  
 4   Contact Name  9994 non-null   object 
 5   Country       9994 non-null   object 
 6   City          9994 non-null   object 
 7   Region        9994 non-null   object 
 8   Subregion     9994 non-null   object 
 9   Customer      9994 non-null   object 
 10  Customer ID   9994 non-null   int64  
 11  Industry      9994 non-null   object 
 12  Segment       9994 non-null   object 
 13  Product       9994 non-null   object 
 14  License       9994 non-null   object 
 15  Sales         9994 non-null   float64
 16  Quantity      9994 non-null   int64  
 17  Discount      9994 non-null   float64
 18  Profit        9994 non-null 

In [4]:
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate Row IDs:", df['Row ID'].duplicated().sum())
print("Unique Order IDs:", df['Order ID'].nunique())
print(df.nunique())

Fully duplicate rows: 0
Duplicate Row IDs: 0
Unique Order IDs: 5009
Row ID          9994
Order ID        5009
Order Date      1237
Date Key        1237
Contact Name     793
Country           48
City             262
Region             3
Subregion         12
Customer          99
Customer ID       99
Industry          10
Segment            3
Product           14
License         9994
Sales           5825
Quantity          14
Discount          12
Profit          7287
dtype: int64


In [5]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
print(df['Order Date'].min(), df['Order Date'].max())

2020-01-04 00:00:00 2023-12-31 00:00:00


In [6]:
cust = df.groupby('Customer ID').agg(
    orders=('Order ID', 'nunique'),
    first_order=('Order Date', 'min'),
    total_profit=('Profit', 'sum')
)
print(cust['orders'].describe())
print(cust['first_order'].dt.to_period('Q').value_counts().sort_index())
print("Customers with total loss:", (cust['total_profit'] < 0).sum())

count     99.000000
mean      50.595960
std       17.537667
min       13.000000
25%       38.500000
50%       52.000000
75%       60.000000
max      103.000000
Name: orders, dtype: float64
first_order
2020Q1    71
2020Q2    22
2020Q3     6
Freq: Q-DEC, Name: count, dtype: int64
Customers with total loss: 10


In [7]:
df.info()
print(df.isnull().sum())
print(df[['Sales','Quantity','Discount','Profit']].describe())
print("Loss-making rows:", (df['Profit'] < 0).sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 19 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   Row ID        9994 non-null   int64         
 1   Order ID      9994 non-null   object        
 2   Order Date    9994 non-null   datetime64[ns]
 3   Date Key      9994 non-null   int64         
 4   Contact Name  9994 non-null   object        
 5   Country       9994 non-null   object        
 6   City          9994 non-null   object        
 7   Region        9994 non-null   object        
 8   Subregion     9994 non-null   object        
 9   Customer      9994 non-null   object        
 10  Customer ID   9994 non-null   int64         
 11  Industry      9994 non-null   object        
 12  Segment       9994 non-null   object        
 13  Product       9994 non-null   object        
 14  License       9994 non-null   object        
 15  Sales         9994 non-null   float64 

In [8]:
cust = df.groupby('Customer ID').agg(
    orders=('Order ID', 'nunique'),
    first_order=('Order Date', 'min'),
    total_profit=('Profit', 'sum')
)
print(cust['orders'].describe())
print(cust['first_order'].dt.to_period('Q').value_counts().sort_index())
print("Customers with total loss:", (cust['total_profit'] < 0).sum())

count     99.000000
mean      50.595960
std       17.537667
min       13.000000
25%       38.500000
50%       52.000000
75%       60.000000
max      103.000000
Name: orders, dtype: float64
first_order
2020Q1    71
2020Q2    22
2020Q3     6
Freq: Q-DEC, Name: count, dtype: int64
Customers with total loss: 10


In [9]:
import numpy as np

df['Margin'] = df['Profit'] / df['Sales']
df['Order_Year'] = df['Order Date'].dt.year
df['Order_Month'] = df['Order Date'].dt.to_period('M').astype(str)
df['Is_Loss'] = (df['Profit'] < 0).astype(int)

bins = [-0.001, 0, 0.10, 0.20, 0.30, 0.40, 1.0]
labels = ['0%', '1-10%', '11-20%', '21-30%', '31-40%', '40%+']
df['Discount_Tier'] = pd.cut(df['Discount'], bins=bins, labels=labels)

print(df[['Margin','Order_Year','Order_Month','Is_Loss','Discount_Tier']].head())
print(df['Discount_Tier'].value_counts().sort_index())
print("Tier blanks:", df['Discount_Tier'].isnull().sum())

   Margin  Order_Year Order_Month  Is_Loss Discount_Tier
0  0.1600        2022     2022-11        0            0%
1  0.3000        2022     2022-11        0            0%
2  0.4700        2022     2022-06        0            0%
3 -0.4000        2021     2021-10        1          40%+
4  0.1125        2021     2021-10        0        11-20%
Discount_Tier
0%        4798
1-10%       94
11-20%    3709
21-30%     227
31-40%     233
40%+       933
Name: count, dtype: int64
Tier blanks: 0


In [10]:
import os
os.makedirs("Data/Clean", exist_ok=True)
df.to_csv("Data/Clean/saas_clean.csv", index=False)
print("Saved", df.shape)

Saved (9994, 24)


## Findings
- 9,994 rows, 19 columns. No duplicate rows or Row IDs.
- Missing values: ___ (check your output)
- 5,009 orders, 99 customers, 3 segments, 14 products, 3 regions
- Dates: 4 Jan 2020 to 31 Dec 2023
- Loss-making order lines: 1,871
- 10 customers have a total loss
- All 99 customers first ordered in 2020, so cohort analysis is replaced by yearly retention
- Repeat-customer rate dropped as a KPI (about 50 orders per customer)
- Discount tiers 1-10%, 21-30% and 31-40% are small (under 250 rows each)
- Extreme values: Sales max 22,638, Profit min -6,600 (kept, they are real orders)